# CIFAR-10 CNN: Tucker-2 rank sweep

`01_tucker2_conv.ipynb` で1つのConv2dをTucker-2へ置き換えられた前提で、
`rank_out` / `rank_in` を複数試し、サイズと精度のtrade-offを比較する。

このNotebookでは新しい分解アルゴリズムを作らない。
01で完成した1層置換処理を再利用する。

ゴールは「どのrankが良いか」を単一指標ではなく、
**パラメータ削減とvalidation accuracyの両方から選べること**。


## 1. baseline・DataLoader・評価関数を準備する

既存の `CIFAR10CNN`、baseline checkpoint、`evaluate`、パラメータ数計測など
`src` にある共通処理を優先して使う。

CIFAR-10 DataLoaderはSVD実験で使った条件と揃える。
新しい前処理を勝手に追加して比較条件を変えない。


In [12]:
from pathlib import Path
import copy

import torch
from torch import nn

from __future__ import annotations

from copy import deepcopy
import copy
from pathlib import Path
import json
import platform
import sys

print("import: numpy / pandas / matplotlib", flush=True)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# 評価
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

# PyTorch
print("import: torch", flush=True)
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from sklearn.preprocessing import MinMaxScaler

# Jupyter の cwd が notebooks/ でも src を見つける
for _candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    _src = _candidate / "src"
    if (_src / "nn_compression").is_dir():
        if str(_src) not in sys.path:
            sys.path.insert(0, str(_src))
        break
# CursorでTucker処理をsrcへ共通化した後、実際のexport先からimportする。
print("import: nn_compression", flush=True)

from nn_compression.compression import (
    build_tucker2_conv,
    hosvd,
    reconstruct_tucker,
    tucker2_decompose_conv_weight,
    tucker_parameter_count,
)
from nn_compression.metrics import (
    relative_frobenius_error,
)

from nn_compression.utils import (
    find_project_root,
    get_experiment_dirs,
    get_named_module,
    set_seed,
)
from nn_compression.training import (
    evaluate,
    fit_with_early_stopping,
    non_shuffling_loader,
    train_one_epoch,
)
from nn_compression.selection import (
    extract_pareto_frontier,
    find_knee_point,
    get_first_point,
    line_equation,
)
from nn_compression.metrics import (
    accuracy_drop,
    agreement,
    benchmark_inference,
    collect_compression_metrics,
    take_inference_batch,
    count_parameters,
    estimate_cnn_macs,
    estimate_conv2d_macs,
    logits_rmse,
    parameters_reduction,
)
from nn_compression.compression import (
    factorize_conv2d_layer,
    factorize_named_layers,
    retained_energy,
    sweep_conv_svd_ranks,
)
from nn_compression.models import CIFAR10CNN
from nn_compression.datasets import shuffled_index_splits
from nn_compression.compression import (
    hosvd,
    reconstruct_tucker,
    tucker_parameter_count,
)
from nn_compression.metrics import (
    relative_frobenius_error,
)
PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT.name != "nn-compression-svd-dmrg" and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

MODEL_PATH = (
    PROJECT_ROOT
    / "models/10_svd/40_cifar10_cnn/02_svd_global_compression_using_src_corrected"
    / "cifar10_cnn_baseline.pt"
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


import: numpy / pandas / matplotlib
import: torch
import: nn_compression


device(type='cuda')

In [13]:
# 再現性のため乱数seedを固定する
SEED = 0
set_seed(SEED)

# CUDA → MPS → CPU の順で利用可能なデバイスを選ぶ
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"device: {device}")
print(f"PyTorch: {torch.__version__}")

device: cuda
PyTorch: 2.11.0+cu128


In [14]:
# method-first の新しい構成に合わせて保存先を識別する。
METHOD_NAME = "20_tucker"
CASE_NAME = "10_cifar10_cnn"
# 原本の results/models を上書きしない
EXPERIMENT_NAME = "02_rank_sweep"

project_root = find_project_root(Path.cwd())
data_dir, models_dir, results_dir = get_experiment_dirs(
    project_root,
    METHOD_NAME,
    CASE_NAME,
    EXPERIMENT_NAME,
)

checkpoint_dir = models_dir
csv_dir = results_dir
figure_dir = results_dir
metadata_dir = results_dir

print("project_root:", project_root)
print("data_dir:", data_dir)
print("models_dir:", models_dir)
print("results_dir:", results_dir)


project_root: D:\dev\nn-compression-svd-dmrg
data_dir: D:\dev\nn-compression-svd-dmrg\data
models_dir: D:\dev\nn-compression-svd-dmrg\models\20_tucker\10_cifar10_cnn\02_rank_sweep
results_dir: D:\dev\nn-compression-svd-dmrg\results\20_tucker\10_cifar10_cnn\02_rank_sweep


In [15]:
import torchvision.transforms as transforms

# 学習データ専用の前処理（Data Augmentation を含む）
#transforms.Compose([...]) は、リスト中の変換を上から順番に連結して実行する仕組みです。
train_transform = transforms.Compose(
    [
        # 元画像（32 x 32）の周囲を4 pixelぶんゼロ埋めして40 x 40にして
        # 領域内からランダムな位置で 32 x 32 を切り出す。
        # 物体の位置が少しずれても認識できるように学習させる。
        transforms.RandomCrop(
            size=32,
            padding=4,
        ),

        # 確率 p=0.5（デフォルト）で画像を左右反転する。
        # CIFAR-10では、左右が反転しても通常はクラスが変わらないため有効。
        # 例: 左向きの車も右向きの車も automobile として学習する。
        transforms.RandomHorizontalFlip(),

        # PIL Image / NumPy配列をPyTorch Tensorに変換する。
        # 形状: (H, W, C) = (32, 32, 3) -> (C, H, W) = (3, 32, 32)
        # 型・値域: uint8 の [0, 255] -> float32 の [0.0, 1.0]
        transforms.ToTensor(),

        # RGB各チャネルを同じ平均・標準偏差で正規化する。
        # 各画素値 x (ToTensor後は [0, 1]) に対し、
        # x_normalized = (x - 0.5) / 0.5 を適用する。
        # その結果、値域は概ね [0, 1] -> [-1, 1] となる。
        transforms.Normalize(
            mean=(0.5, 0.5, 0.5),  # R, G, B の平均との差し引き用
            std=(0.5, 0.5, 0.5),   # R, G, B のスケール調整用
        ),
    ]
)


# 検証・テスト専用の前処理
evaluation_transform = transforms.Compose(
    [
        # 評価時もCNNに渡せるTensor形式へ変換する。
        # (32, 32, 3) -> (3, 32, 32)、[0, 255] -> [0.0, 1.0]
        transforms.ToTensor(),

        # 学習時と「まったく同じ」正規化を行う。
        # 学習と評価でスケールが違うと、入力分布が変わって正しい評価にならない。
        transforms.Normalize(
            mean=(0.5, 0.5, 0.5),
            std=(0.5, 0.5, 0.5),
        ),
    ]
)

In [16]:
# ============================================================
# CIFAR-10を最初の1回だけダウンロードする
# ============================================================

full_train_augmented = datasets.CIFAR10(
    root=data_dir,
    train=True,
    # 今回はミラーサイトから手動でDLしたためFalse
    download=False,
    transform=train_transform,
)

# すでにダウンロード済みなので download=False
full_train_evaluation = datasets.CIFAR10(
    root=data_dir,
    train=True,
    download=False,
    transform=evaluation_transform,
)

# testデータも同じCIFAR-10アーカイブ内に含まれている
test_dataset = datasets.CIFAR10(
    root=data_dir,
    train=False,
    download=False,
    transform=evaluation_transform,
)

TRAIN_SIZE = 40_000
VALIDATION_SIZE = 5_000

# ------------------------------------------------------------
# train / validation のindexを固定
# SEED が同じなら randperm 結果も同一になり、割当を再現できる。
# 【.py】datasets/splits.py。分割長だけ実験条件
# ------------------------------------------------------------
train_indices, validation_indices_early_stop, validation_indices_rank = (
    shuffled_index_splits(
        len(full_train_augmented),
        (TRAIN_SIZE, VALIDATION_SIZE, VALIDATION_SIZE),
        seed=SEED,
    )
)

# ------------------------------------------------------------
# 同じ元データだがtransformを変える
# ------------------------------------------------------------

# train: augmentationあり
train_dataset = Subset(
    dataset=full_train_augmented,
    indices=train_indices,
)

# validation: augmentationなし
validation_dataset_early_stop = Subset(
    dataset=full_train_evaluation,
    indices=validation_indices_early_stop,
)

validation_dataset_rank = Subset(
    dataset=full_train_evaluation,
    indices=validation_indices_rank,
)

print(f"train:      {len(train_dataset):,}")
print(f"validation: {len(validation_dataset_early_stop):,}")
print(f"validation: {len(validation_dataset_rank):,}")
print(f"test:       {len(test_dataset):,}")


train:      40,000
validation: 5,000
validation: 5,000
test:       10,000


In [17]:
BATCH_SIZE = 256
NUM_WORKERS = 0
# shuffleの順序も再現しやすくする
loader_generator = torch.Generator().manual_seed(SEED)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda"),
    generator=loader_generator,
)
# corrected: shuffle=False の train 再評価用。本実験は reevaluate_train=False のため fit には渡さない
train_eval_loader = non_shuffling_loader(train_loader)

validation_loader_early_stop = DataLoader(
    validation_dataset_early_stop,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda"),
)

validation_loader_rank = DataLoader(
    validation_dataset_rank,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda"),
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda"),
)

# train_loaderをnext(iter(...))するとshuffle用Generatorを消費するので、
# shape確認だけならDatasetから1件直接取り出す
image, label = train_dataset[0]

print(f"image shape: {tuple(image.shape)}")
print(f"label: {label}")
print(
    "expected batch shape: "
    f"({BATCH_SIZE}, {image.shape[0]}, {image.shape[1]}, {image.shape[2]})"
)

image shape: (3, 32, 32)
label: 6
expected batch shape: (256, 3, 32, 32)


## 2. 01で完成したTucker-2 Conv置換処理を使えるようにする

01の実装がまだNotebook内だけなら一時的にコピーしてよい。

ただし、このNotebookでは置換ロジックを書き直さない。
後で `conv_tucker.py` に共通化したらimportへ置き換える。


## 3. 比較するrank候補を決める

対象はまず `conv2` 1層だけに固定する。

`rank_out` と `rank_in` を同時に変えると組合せが増えるため、
最初は少数の候補から始める。

各rankは元の `out_channels` / `in_channels` を超えないこと。


In [18]:
model = CIFAR10CNN().to(device)

# 01_cnn_cifar10_baseline: {"model_state_dict": ...} のメタ付きdict
# 02_svd_..._corrected: state_dict をそのまま保存
checkpoint = torch.load(MODEL_PATH, map_location=device, weights_only=False)
# 読み込んだものが辞書で、かつ "model_state_dict" というキーを持つか？
if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
    state_dict = checkpoint["model_state_dict"]
else:
    state_dict = checkpoint

model.load_state_dict(state_dict)
model.eval()

target_conv_for_print = model.conv2

print("loaded:", MODEL_PATH)
print("weight shape:", tuple(target_conv_for_print.weight.shape))
print("bias:", target_conv_for_print.bias is not None)
print("stride:", target_conv_for_print.stride)
print("padding:", target_conv_for_print.padding)
print("groups:", target_conv_for_print.groups)


partial_rank_settings = [
    # 例を参考に、自分で比較したい(rank_out, rank_in)を数組決める。
    #{"rank_out": ..., "rank_in": ...},
    {"rank_out": 64, "rank_in": 32},
    {"rank_out": 64, "rank_in": 24},
    {"rank_out": 64, "rank_in": 16},
    {"rank_out": 64, "rank_in": 8},
    {"rank_out": 48, "rank_in": 32},
    {"rank_out": 48, "rank_in": 24},
    {"rank_out": 48, "rank_in": 16},
    {"rank_out": 48, "rank_in": 8},
    {"rank_out": 32, "rank_in": 32},
    {"rank_out": 32, "rank_in": 24},
    {"rank_out": 32, "rank_in": 16},
    {"rank_out": 32, "rank_in": 8},
    {"rank_out": 16, "rank_in": 32},
    {"rank_out": 16, "rank_in": 24},
    {"rank_out": 16, "rank_in": 16},
    {"rank_out": 16, "rank_in": 8},
    {"rank_out": 8, "rank_in": 32},
    {"rank_out": 8, "rank_in": 24},
    {"rank_out": 8, "rank_in": 16},
    {"rank_out": 8, "rank_in": 8},
]


loaded: d:\dev\nn-compression-svd-dmrg\models\10_svd\40_cifar10_cnn\02_svd_global_compression_using_src_corrected\cifar10_cnn_baseline.pt
weight shape: (64, 32, 3, 3)
bias: True
stride: (1, 1)
padding: (1, 1)
groups: 1


## 4. 各rankで同じ実験を繰り返す

各候補について必ず同じ順番で比較する。

1. baselineをdeepcopy
2. `conv2` だけTucker-2へ置換
3. Tucker-2後のパラメータ数を数える
4. validation accuracyを測る
5. baselineからのaccuracy dropを求める
6. 結果を1行として保存する

最低限保存する列:

```text
rank_out
rank_in
parameters
parameters_reduction
validation_acc
accuracy_drop
```

必要ならweight再構成誤差も追加する。


In [19]:
# Tucker-2 Conv の理論 MACs。
# SVD の estimate_cnn_macs / compressed_conv2d_macs に対応する Notebook 側ヘルパー。
# build_tucker2_conv と同じ 1x1 -> kHxkW -> 1x1 の積和で見積もる。

from nn_compression.metrics.macs import conv2d_macs, linear_macs, compressed_linear_macs

# CIFAR-10 CNN（pool 前の Conv 出力）。SVD 02 と同じ。
CIFAR10_CONV_OUTPUT_HW = {
    "conv1": (32, 32),
    "conv2": (16, 16),
    "conv3": (8, 8),
}
CIFAR10_LINEAR_LAYER_NAMES = ("fc1", "fc2")


def compressed_tucker2_conv2d_macs(
    conv: nn.Conv2d,
    rank_out: int,
    rank_in: int,
    out_h: int,
    out_w: int,
) -> int:
    """Tucker-2 置換後の 1 層分 MACs。

    1x1: Cin -> rank_in
    kHxkW: rank_in -> rank_out
    1x1: rank_out -> Cout
    空間サイズは分解前の Conv 出力と同じ（stride=1 前提）。

    Args:
        conv: 圧縮前の nn.Conv2d（weight.shape から Cin/Cout/kH/kW を取る）
        rank_out: out channel 側の Tucker rank
        rank_in: in channel 側の Tucker rank
        out_h: Conv 出力の高さ（CIFAR conv2 なら 16）
        out_w: Conv 出力の幅（CIFAR conv2 なら 16）

    Returns:
        圧縮後 1 層の理論 MACs（int）
    """
    out_ch, in_ch, k_h, k_w = conv.weight.shape
    input_macs = out_h * out_w * rank_in * in_ch
    core_macs = out_h * out_w * rank_out * rank_in * k_h * k_w
    output_macs = out_h * out_w * out_ch * rank_out
    return input_macs + core_macs + output_macs


def estimate_tucker2_conv2d_macs(
    conv: nn.Conv2d,
    rank_out: int,
    rank_in: int,
    out_hw: tuple[int, int],
    verbose: bool = True,
) -> tuple[int, int, float]:
    """1 つの Conv2d を Tucker-2 にしたときの MACs と削減率。

    他層は含めない。比較対象はその 1 層のみ。
    削減率は ``1 - compressed / baseline``。

    Args:
        conv: 圧縮前の nn.Conv2d
        rank_out: out channel 側の Tucker rank
        rank_in: in channel 側の Tucker rank
        out_hw: Conv 出力空間サイズ ``(out_h, out_w)``
        verbose: True なら baseline / compressed / 削減率を print

    Returns:
        (baseline_macs, compressed_macs, compute_reduction)
        - baseline_macs: 元 Conv の MACs
        - compressed_macs: Tucker-2 後の MACs
        - compute_reduction: 削減率（0〜1、大きいほど削減大）
    """
    out_h, out_w = out_hw
    baseline_macs = conv2d_macs(conv, out_h, out_w)
    compressed_macs = compressed_tucker2_conv2d_macs(
        conv, rank_out, rank_in, out_h, out_w
    )
    compute_reduction = 1.0 - compressed_macs / baseline_macs

    if verbose:
        print("Baseline MACs:", baseline_macs)
        print("Compressed MACs:", compressed_macs)
        print(f"Compute reduction: {compute_reduction:.2%}")

    return baseline_macs, compressed_macs, compute_reduction


def estimate_cnn_tucker2_conv_macs(
    model,
    rank_out: int,
    rank_in: int,
    verbose: bool = True,
    *,
    layer_name: str = "conv2",
    out_hw: tuple[int, int] | None = None,
) -> tuple[int, int, float]:
    """指定 Conv 層だけを Tucker-2 したときの MACs・削減率。

    SVD の ``estimate_cnn_conv2_macs`` に対応。省略時は CIFAR-10 の
    ``conv2`` / 16×16。

    Args:
        model: 圧縮前の CNN（named module から層を取る）
        rank_out: out channel 側の Tucker rank
        rank_in: in channel 側の Tucker rank
        verbose: True なら結果を print
        layer_name: 対象層名（既定 ``"conv2"``）
        out_hw: 出力空間サイズ。None なら ``CIFAR10_CONV_OUTPUT_HW[layer_name]``

    Returns:
        (baseline_macs, compressed_macs, compute_reduction)
        いずれも「その 1 層だけ」の値。CNN 全体ではない。
    """
    if out_hw is None:
        out_hw = CIFAR10_CONV_OUTPUT_HW[layer_name]
    conv = get_named_module(model, layer_name)
    return estimate_tucker2_conv2d_macs(
        conv, rank_out, rank_in, out_hw, verbose=verbose
    )


def estimate_cnn_tucker2_macs(
    model,
    rank_out: int | None = None,
    rank_in: int | None = None,
    verbose: bool = True,
    *,
    conv_tucker_ranks: dict[str, tuple[int, int]] | None = None,
    linear_ranks: dict[str, int] | None = None,
    conv_output_hw: dict[str, tuple[int, int]] | None = None,
    linear_layer_names: tuple[str, ...] = CIFAR10_LINEAR_LAYER_NAMES,
) -> tuple[int, int, float]:
    """CNN 全体の MACs・削減率を Tucker-2 圧縮前提で比較する。

    ``conv_output_hw`` の Conv と ``linear_layer_names`` の Linear を合計する。
    ``conv_tucker_ranks`` にある層だけ Tucker-2（rank_out, rank_in）として数え、
    他の Conv / Linear は元のまま。``linear_ranks`` は SVD 同様の 2 層 Linear。

    位置引数 ``rank_out`` / ``rank_in`` だけ渡すと、この実験の既定どおり
    ``conv2`` のみ Tucker-2 圧縮として扱う。

    Args:
        model: 圧縮前の CNN
        rank_out: conv2 だけの短縮指定用（``conv_tucker_ranks`` 未指定時）
        rank_in: conv2 だけの短縮指定用（``conv_tucker_ranks`` 未指定時）
        verbose: True なら全体 MACs と削減率を print
        conv_tucker_ranks: ``{層名: (rank_out, rank_in)}``。指定時はこちら優先
        linear_ranks: ``{層名: rank}``。省略時は Linear を圧縮しない
        conv_output_hw: ``{層名: (out_h, out_w)}``。省略時は CIFAR-10 既定
        linear_layer_names: 合計に含める Linear 名（既定 fc1, fc2）

    Returns:
        (baseline_macs, compressed_macs, reduction)
        - baseline_macs: 全対象層の合計 MACs（圧縮前）
        - compressed_macs: Tucker-2 / 低ランク Linear を反映した合計 MACs
        - reduction: 全体の削減率 ``1 - compressed / baseline``
    """
    if conv_output_hw is None:
        conv_output_hw = CIFAR10_CONV_OUTPUT_HW
    if conv_tucker_ranks is None:
        if rank_out is None or rank_in is None:
            raise TypeError(
                "rank_out と rank_in、または conv_tucker_ranks を指定してください。"
            )
        conv_tucker_ranks = {"conv2": (rank_out, rank_in)}
    if linear_ranks is None:
        linear_ranks = {}

    baseline_macs = 0
    compressed_macs = 0

    for layer_name, out_hw in conv_output_hw.items():
        conv = get_named_module(model, layer_name)
        out_h, out_w = out_hw
        layer_baseline = conv2d_macs(conv, out_h, out_w)
        baseline_macs += layer_baseline
        if layer_name in conv_tucker_ranks:
            r_out, r_in = conv_tucker_ranks[layer_name]
            compressed_macs += compressed_tucker2_conv2d_macs(
                conv, r_out, r_in, out_h, out_w
            )
        else:
            compressed_macs += layer_baseline

    for layer_name in linear_layer_names:
        linear = get_named_module(model, layer_name)
        layer_baseline = linear_macs(linear)
        baseline_macs += layer_baseline
        if layer_name in linear_ranks:
            compressed_macs += compressed_linear_macs(
                linear.in_features,
                linear.out_features,
                linear_ranks[layer_name],
            )
        else:
            compressed_macs += layer_baseline

    reduction = 1.0 - compressed_macs / baseline_macs

    if verbose:
        print("Baseline total MACs:", baseline_macs)
        print("Compressed total MACs:", compressed_macs)
        print(f"Compute reduction: {reduction:.2%}")

    return baseline_macs, compressed_macs, reduction


# 動作確認例:
# estimate_cnn_tucker2_conv_macs(model, rank_out=32, rank_in=16)  # conv2 層だけ
# estimate_cnn_tucker2_macs(model, rank_out=32, rank_in=16)       # CNN 全体


In [25]:
results = []
criterion = nn.CrossEntropyLoss()


baseline_loss, baseline_acc = evaluate(
    model,                      # 圧縮前
    validation_loader_rank,     # rank 比較用 validation（Early Stopping 用とは別）
    criterion,
    device,
)
for setting in partial_rank_settings:
    #前の compressed_model は次のiterationで参照されなくなるので順次解放対象になります。
    compressed_model = copy.deepcopy(model)
    target_conv=compressed_model.conv2
    rank_out = setting["rank_out"]
    rank_in = setting["rank_in"]
    core, u_out, u_in = tucker2_decompose_conv_weight(
        target_conv.weight.detach(), rank_out, rank_in
    )
    tucker_conv = build_tucker2_conv(target_conv, rank_out, rank_in).to(device)
    compressed_model.conv2=tucker_conv
    validation_loss, validation_acc = evaluate(
        compressed_model,
        validation_loader_rank,
        criterion,
        device,
    )

    reconstructed_weight = reconstruct_tucker(
        core,
        {
            0: u_out,
            1: u_in,
        },
    )

    weight_relative_error = relative_frobenius_error(
        target_conv.weight.detach(),
        reconstructed_weight,
    )
    _,_, conv2_reduction = estimate_cnn_tucker2_conv_macs(
        model, rank_out, rank_in, verbose=False
    )

    _,_, all_reduction = estimate_cnn_tucker2_macs(
        model,
        rank_out=rank_out,
        rank_in=rank_in,
        verbose=False,
    )

    results.append({
        "rank_out":rank_out,
        "rank_in":rank_in,
        "parameters":count_parameters(compressed_model),
        "conv2 parameters":tucker_parameter_count(
            tuple(target_conv.weight.shape),
            {0: rank_out, 1: rank_in},
        ),
        "parameters_reduction":parameters_reduction(model,compressed_model),
        "validation_acc":validation_acc,
        "validation_loss":validation_loss,
        "accuracy_drop":baseline_acc - validation_acc,
        "conv2_macs_reduction":conv2_reduction,
        "all_macs_reduction":all_reduction,
        "weight_relative_error": weight_relative_error.item(),
    })


## 5. 表にしてtrade-offを読む

DataFrameなどでrank候補を横並びにする。

「一番accuracyが高い」だけでも「一番小さい」だけでもなく、

```text
パラメータを大きく減らせる
かつ
accuracy dropが許容できる
```

候補を探す。


In [ ]:
import pandas as pd

df = pd.DataFrame(results)

results_dir.mkdir(parents=True, exist_ok=True)
rank_sweep_csv_path = results_dir / "rank_sweep_results.csv"
df.to_csv(rank_sweep_csv_path, index=False)
print("saved:", rank_sweep_csv_path)

df


## 6. 可能ならPareto的に候補を見る

ある候補Aが候補Bより

- パラメータ数が少ない
- accuracyも高い

なら、Bを選ぶ理由は弱い。

最終的にfine-tuningへ持っていくrank候補を1〜数個選ぶ。
複雑な自動rank選択アルゴリズムはまだ作らなくてよい。

Paretoの意味（短い補足）: AがBよりparametersもvalidation_lossも悪くないうえ、少なくとも一方で良い場合、BはAに支配される。支配されない点だけが frontier に残る。


In [ ]:
# Section 5 の df を入力に Pareto frontier を見る。
# 判定本体は extract_pareto_frontier（src）。ここはフィルタ・表示・プロット・保存。
from IPython.display import display

baseline_params = count_parameters(model)

# baseline よりパラメータが多い候補は圧縮として除外（SVD版と同趣旨）。
# MACs / compute_reduction は使わない。
df_pareto_input = df[df["parameters"] <= baseline_params].copy()

df_pareto = extract_pareto_frontier(
    df_pareto_input,
    x_column="parameters",
    y_column="validation_loss",
)

pareto_view_columns = [
    "rank_out",
    "rank_in",
    "parameters",
    "parameters_reduction",
    "validation_loss",
    "validation_acc",
    "accuracy_drop",
]
pareto_view_columns = [c for c in pareto_view_columns if c in df_pareto.columns]

print("baseline_params:", baseline_params)
print("pareto input rows:", len(df_pareto_input))
print("pareto frontier points:", len(df_pareto))
display(df_pareto[pareto_view_columns])

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(
    df["parameters"],
    df["validation_loss"],
    label="all candidates",
    alpha=0.7,
)
ax.scatter(
    df_pareto["parameters"],
    df_pareto["validation_loss"],
    label="Pareto frontier",
    zorder=3,
)
ax.plot(
    df_pareto["parameters"],
    df_pareto["validation_loss"],
    linestyle="--",
    linewidth=1,
    zorder=2,
)
ax.set_xlabel("Parameters")
ax.set_ylabel("Validation loss")
ax.set_title("Tucker-2 rank sweep Pareto frontier")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

pareto_csv_path = results_dir / "pareto_frontier.csv"
df_pareto.to_csv(pareto_csv_path, index=False)
print("saved:", pareto_csv_path)


## 6.1 fine-tuning 用 3 候補（aggressive / balanced / conservative）

Pareto frontier（parameters 昇順）上で、SVD版と同じく正規化座標の knee を求める。

- **balanced**: knee（端点を結ぶ直線から最も遠い点）
- **aggressive**: knee より parameters が少ない側（より高圧縮）の隣接点
- **conservative**: knee より parameters が多い側（より低圧縮）の隣接点

knee が端点で片側に隣が無いときだけ、frontier 上の別の最も近い点で 3 候補が重複しないように補う。
rank はハードコードしない。sweep 結果が変われば選択も自動で変わる。


In [ ]:
# SVD 02 combination knee と同じ手順。
# 【.py】get_first_point / line_equation / find_knee_point
# 【.ipynb】3 役割への割り当てと CSV 名

df_pareto_view = df_pareto.sort_values("parameters").reset_index(drop=True).copy()

if len(df_pareto_view) < 3:
    raise ValueError(
        f"fine-tuning 用に Pareto 点が 3 つ以上必要です（現在 {len(df_pareto_view)}）。"
    )

scaler = MinMaxScaler()
df_pareto_view[
    ["parameters_normalize", "validation_loss_normalize"]
] = scaler.fit_transform(
    df_pareto_view[["parameters", "validation_loss"]]
)

p0 = get_first_point(
    df_pareto_view,
    "parameters_normalize",
    "parameters_normalize",
    "validation_loss_normalize",
)
p1 = get_first_point(
    df_pareto_view,
    "validation_loss_normalize",
    "parameters_normalize",
    "validation_loss_normalize",
)
knee_point, knee_distance = find_knee_point(
    df_pareto_view,
    *line_equation(p0, p1),
    "parameters_normalize",
    "validation_loss_normalize",
)

knee_mask = (
    (df_pareto_view["parameters_normalize"] == knee_point[0])
    & (df_pareto_view["validation_loss_normalize"] == knee_point[1])
)
knee_pos = int(df_pareto_view.index[knee_mask][0])
n_pareto = len(df_pareto_view)

balanced_pos = knee_pos
aggressive_pos = knee_pos - 1
conservative_pos = knee_pos + 1


def _nearest_unused(center: int, used: set[int]) -> int:
    """未使用点のうち center に最も近い index を返す。"""
    candidates = sorted(
        (abs(i - center), i) for i in range(n_pareto) if i not in used
    )
    if not candidates:
        raise ValueError("Pareto 上に未使用点がありません。")
    return candidates[0][1]


used = {balanced_pos}
if aggressive_pos < 0:
    aggressive_pos = _nearest_unused(balanced_pos, used)
used.add(aggressive_pos)
if conservative_pos >= n_pareto:
    conservative_pos = _nearest_unused(balanced_pos, used)
used.add(conservative_pos)

selected_positions = {
    "aggressive": int(aggressive_pos),
    "balanced": int(balanced_pos),
    "conservative": int(conservative_pos),
}
if len(set(selected_positions.values())) < 3:
    raise ValueError(
        f"3 候補が重複しています: {selected_positions} / n={n_pareto}, knee_pos={knee_pos}"
    )

selected_rows = []
for role in ("aggressive", "balanced", "conservative"):
    row = df_pareto_view.iloc[selected_positions[role]].to_dict()
    row["role"] = role
    selected_rows.append(row)

selected_rank_settings_df = pd.DataFrame(selected_rows)
preferred_columns = [
    "role",
    "rank_out",
    "rank_in",
    "parameters",
    "parameters_reduction",
    "validation_loss",
    "validation_acc",
    "accuracy_drop",
    "conv2_macs_reduction",
    "all_macs_reduction",
    "weight_relative_error",
]
ordered = [c for c in preferred_columns if c in selected_rank_settings_df.columns]
extra = [c for c in selected_rank_settings_df.columns if c not in ordered]
selected_rank_settings_df = selected_rank_settings_df[ordered + extra]

print(f"knee distance: {knee_distance:.6f}")
print(f"knee_pos={knee_pos}, selected_positions={selected_positions}")
print("\nSelected Tucker-2 ranks:")
for _, row in selected_rank_settings_df.iterrows():
    print(
        f"  {row['role']:12s}: rank_out={int(row['rank_out'])}, "
        f"rank_in={int(row['rank_in'])}, "
        f"params={int(row['parameters'])}, "
        f"val_loss={row['validation_loss']:.4f}"
    )

display(selected_rank_settings_df[preferred_columns])

selected_csv_path = results_dir / "selected_rank_settings.csv"
selected_rank_settings_df[preferred_columns].to_csv(selected_csv_path, index=False)
print("saved:", selected_csv_path)


## 7. このNotebookの完了条件

次を説明できれば `03_finetuning.ipynb` へ進む。

1. `rank_out` / `rank_in` を下げると何が小さくなるか
2. rankを下げすぎるとaccuracyが落ちる理由
3. パラメータ数だけでrankを選んではいけない理由
4. fine-tuningへ持っていくrank候補を根拠付きで選べる
